<img src="../assets/logo-banner.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-light">
<img src="../assets/logo-banner-dark.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-dark">

# radar datatree: Cloud-native, time-aware weather radar datasets

You want the radar data for a Chicago-area storm. The traditional path: find the right NEXRAD Level II files, download gigabytes of them, and decode each one before you can look at anything. **This notebook is about reaching the archive: connect, inspect the hierarchy, read back one scan, and plot it.**

Nothing here downloads a bulk archive or writes a file.

## Why a DataTree?

Weather radar volumes are organized hierarchically: a Volume Coverage Pattern (VCP) cycles through several 360° sweeps at increasing elevation angles, repeating every 4–10 minutes. Different VCPs have different shapes, which is exactly what `xarray.DataTree` is built for.

- **Hierarchical** — every VCP × sweep combination is a node you navigate (`dt["VCP-12/sweep_0"]`), not a separate file you parse.
- **Time-indexed** — all scans share a `vcp_time` dimension, so `.sel(vcp_time=...)` replaces the file-iteration loop.
- **Lazy** — opening the full archive fetches metadata only (~MB). Variables stream from object storage on demand.

For architecture and scaling benchmarks, see *Ladino-Rincón et al.* (2026, submitted to *IEEE Transactions on Big Data*). Unfamiliar acronyms (VCP, polarimetric, ZDR, RHOHV, ...) are all defined in the [glossary](glossary).

## radar datatree in practice

Let's see how the radar datatree looks in practice.

In [ ]:
# Colab bootstrap — runs only on Google Colab; no-op locally and in CI.
import sys

if "google.colab" in sys.modules:
    %pip install -qqq icechunk "rustytree-xarray>=0.3.0" "xradar>=0.12.0" \
        "zarr>=3.1.2" "s3fs>=2025.5.1" cmweather

In [ ]:
import icechunk as ic

# Anonymous, read-only session against the public KLOT archive on AWS Open
# Data. `anonymous=True` is what makes the bucket readable without
# credentials — drop it and reads fail with an auth error even though the
# bucket is public.
storage = ic.s3_storage(
    bucket="nexrad-arco", prefix="KLOT", region="us-east-1", anonymous=True
)
session = ic.Repository.open(storage).readonly_session("main")
print("Connected to s3://nexrad-arco/KLOT on branch 'main'")

We can use [`xarray.open_datatree`](https://docs.xarray.dev/en/stable/generated/xarray.open_datatree.html) to explore the radar archive — with `engine="rustytree"`, a Rust-backed xarray DataTree backend recommended for radar-datatree archives. It's a drop-in replacement for the standard `engine="zarr"`, ~10× faster on icechunk repos served from object storage (see [`rustytree-xarray` on PyPI](https://pypi.org/project/rustytree-xarray/)).

In [ ]:
import xarray as xr
import xradar  # noqa: F401  — registers the .xradar accessor

dt = xr.open_datatree(session.store, engine="rustytree", chunks=None)
dt

We can access any of these VCPs using a file-path syntax — for example, `dt["VCP-212/sweep_0"]` — which returns an `xarray.Dataset`. Before slicing anything though, let's see how big the full archive is.

In [ ]:
print(f"datatree size: {dt.nbytes / 1024**4:.2f} TB")

That is the **logical** size — the uncompressed extent of every array in the archive, which is far larger than the compressed bytes actually stored, and larger still than what any one query transfers. It grows every time the radar completes a volume, so the number above moves. Either way it is too much to hold in a session, and we do not have to: we can open only the sweeps or VCPs we care about.

## Opening only what you need

`xarray` and `zarr` let us inspect a dataset's structure by pulling metadata only — no data is read until we ask for it. `rustytree` gives us two ways to slice the archive at open time:

- **`group_filter`** takes a glob and returns a *filtered DataTree* — the matching nodes plus their parent VCP groups, auto-included as ancestors. `"/*/sweep_0"` returns the lowest-elevation cut (`sweep_0`) from every VCP.
- **`group`** takes one exact node path and returns a single *Dataset* via `xr.open_dataset` — the leanest read when you want just one node's arrays.

In [ ]:
dt_sweep0 = xr.open_datatree(
    session.store, engine="rustytree", group_filter="/*/sweep_0"
)

In [ ]:
dt_sweep0

Or pull a **single VCP and sweep** as a plain Dataset with `group`. Here we open **VCP-212** — NEXRAD's severe-weather rapid-scan mode — at its lowest elevation. `open_dataset` returns just that node's arrays: lean and fast, but *without* the coordinates inherited from ancestor nodes (the radar's lat/lon, the `vcp_time` index) — note the short coordinate list below.

In [ ]:
# Pull a single node as a plain Dataset with `group` — the leanest read.
ds_212 = xr.open_dataset(
    session.store,
    engine="rustytree",
    group="/VCP-212/sweep_0",
)

In [ ]:
ds_212

## Selecting one moment in time

Everything so far has been metadata. `.sel()` on `vcp_time` is what finally reads bytes — and only the bytes for the scan you asked for.

We target **18 July 2016, 05:35 UTC**, when an organized line of overnight thunderstorms swept across the Chicago area. It is a fixed historical timestamp, so this notebook returns the same scan every time it runs.

In [ ]:
# Reopen VCP-212/sweep_0 as a filtered DataTree so the scan carries the
# coordinates georeferencing needs (radar lat/lon, range, azimuth, vcp_time).
dt_212 = xr.open_datatree(
    session.store, engine="rustytree", group_filter="/VCP-212/sweep_0"
)
scan_ds = dt_212["VCP-212/sweep_0"].to_dataset(inherit="all_coords")

# method="nearest" snaps to the real VCP cadence — an exact match would raise
# if the radar happened not to be scanning at that instant. georeference()
# then derives Cartesian x, y, z so the polar gates are plottable.
scan = scan_ds.sel(vcp_time="2016-07-18 05:35", method="nearest").xradar.georeference()
scan

## Plot the scan

The data is analysis-ready, so plotting is ordinary `xarray`. Here is horizontal reflectivity (`DBZH`) — the moment that shows where the precipitation is and how heavy it is.

The other polarimetric moments (`ZDR`, `RHOHV`, `PHIDP`) come with the same scan; [Notebook 2](2.KLOT-LowSweeps) plots the full four-panel view and explains what each one tells you.

In [ ]:
import cmweather  # noqa: F401  — registers the ChaseSpectral colormap
import matplotlib.pyplot as plt

# x/y come out of georeference() in metres; rescale so the axes read in km.
scan_km = scan.assign_coords(x=scan.x / 1000, y=scan.y / 1000)

fig, ax = plt.subplots(figsize=(7.5, 6))
scan_km["DBZH"].plot(
    ax=ax,
    x="x",
    y="y",
    cmap="ChaseSpectral",
    vmin=-10,
    vmax=70,
    cbar_kwargs={"label": "Reflectivity [dBZ]"},
)
ax.set(
    title=f"KLOT reflectivity — {str(scan.vcp_time.values)[:19]} UTC",
    xlabel="East-West distance [km]",
    ylabel="North-South distance [km]",
    xlim=(-150, 150),
    ylim=(-150, 150),
)
ax.set_aspect("equal")

## Where to next

That is the whole access story: **connect → open → narrow → select → plot**. One `xarray.Dataset`, read straight from object storage, with no file to manage.

**Want the full polarimetric picture?**
→ [Notebook 2 — KLOT low sweeps](2.KLOT-LowSweeps) grabs `sweep_0` across every VCP two ways — glob-and-concatenate, or open the pre-stitched `KLOT-lowsweeps` virtual archive — then plots Z, ZDR, RHOHV and PHIDP together.

**Want to reproduce a published figure?**
→ [Notebook 3 — QVP comparison](3.QVP-Workflow-Comparison) reproduces Ryzhkov et al. (2016) Fig. 4 and asserts numerical equivalence between the traditional file-based path and the ARCO streaming path.

**Want to estimate rainfall accumulation?**
→ [Notebook 4 — QPE scaling](4.QPE-Scaling-Benchmark) applies the Marshall–Palmer Z–R relation live for one day, with cluster-recommended templates for 7-day / 30-day / 6-month windows.

**Want to query a different radar?**
→ The [quickstart](quickstart) shows the 5-line connection pattern. Swap `"KLOT"` for `"KVNX"` (Oklahoma) to point at a different archive.

**Want to understand the data model?**
→ [About](about) covers the DataTree / Icechunk / Zarr stack and the [AtmoScale](https://atmoscale.ai) parent platform. [Glossary](glossary) defines every radar acronym in one place.

---

*Cite this work:* Ladino-Rincón, A., et al. (2026). *Radar DataTree: A Cloud-Native AI-Ready Data Model for Accessible, Time-Aware Weather Radar Datasets.* Submitted to *IEEE Transactions on Big Data*. Earlier preprint: arXiv:2510.24943, [doi:10.48550/arXiv.2510.24943](https://doi.org/10.48550/arXiv.2510.24943).